In [1]:
import json
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from fetch_utils import get_json

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

START = "2023-01-01"
END = "2026-10-31"


def save_json(data, filename):
    path = RAW / filename
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  Saved {path.relative_to(ROOT)}")


print("Project root:", ROOT)
print("Raw data directory:", RAW.relative_to(ROOT))

Project root: /Users/rhea/Desktop/northbound_project
Raw data directory: data/raw


In [2]:
FRANKFURTER = "https://api.frankfurter.dev/v1"


def frankfurter_url(year):
    start = START if year == 2023 else f"{year}-01-01"
    end = END if year == 2026 else f"{year}-12-31"
    return f"{FRANKFURTER}/{start}..{end}"


fx_rows = []
for year in [2023, 2024, 2025, 2026]:
    data = get_json(frankfurter_url(year), params={"base": "HKD", "symbols": "CNY"})
    if data is None:
        print(f"{year} download failed. Check the connection and rerun this cell.")
        continue
    save_json(data, f"frankfurter_hkd_cny_{year}.json")
    for date, rates in data["rates"].items():
        fx_rows.append({"date": date, "hkd_cny": rates["CNY"]})
    print(f"{year}: {len(data['rates'])} working days")
    time.sleep(1)

  Saved data/raw/frankfurter_hkd_cny_2023.json
2023: 256 working days
  Saved data/raw/frankfurter_hkd_cny_2024.json
2024: 257 working days
  Saved data/raw/frankfurter_hkd_cny_2025.json
2025: 256 working days
  Saved data/raw/frankfurter_hkd_cny_2026.json
2026: 214 working days


In [3]:
fx = pd.DataFrame(fx_rows)
fx["date"] = pd.to_datetime(fx["date"])
fx = fx.sort_values("date")

print("Rows before cleaning:", len(fx))
print("Repeated dates:", fx.loc[fx["date"].duplicated(), "date"].dt.date.tolist())
fx = fx.drop_duplicates("date")
fx = fx[(fx["date"] >= START) & (fx["date"] <= END)]
print("Rows after cleaning:", len(fx))

print("\nDate range:", fx["date"].min().date(), "to", fx["date"].max().date())
print("Duplicate dates:", fx["date"].duplicated().sum())
print("Exchange rate range:", fx["hkd_cny"].min(), "to", fx["hkd_cny"].max(), "(expected: 0.8–1.0)")
print(fx.head())

fx.to_csv(RAW / "fx_frankfurter_daily.csv", index=False)
print("\nSaved data/raw/fx_frankfurter_daily.csv")


Rows before cleaning: 983
Repeated dates: [datetime.date(2023, 12, 29), datetime.date(2024, 12, 31), datetime.date(2025, 12, 31)]
Rows after cleaning: 979

Date range: 2023-01-02 to 2026-10-30
Duplicate dates: 0
Exchange rate range: 0.84797 to 0.94785 (expected: 0.8–1.0)
        date  hkd_cny
1 2023-01-02  0.88379
2 2023-01-03  0.88435
3 2023-01-04  0.88078
4 2023-01-05  0.88012
5 2023-01-06  0.87853

Saved data/raw/fx_frankfurter_daily.csv


In [4]:
load_dotenv(ROOT / ".env")
FRED_KEY = os.getenv("FRED_API_KEY")
print("FRED key loaded:", FRED_KEY is not None, "| length:", len(FRED_KEY or ""), "(expected: 32)")

FRED_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_SERIES = {"DEXCHUS": "cny_per_usd", "DEXHKUS": "hkd_per_usd"}

fred_tables = []
for series_id, column in FRED_SERIES.items():
    params = {
        "series_id": series_id,
        "api_key": FRED_KEY,
        "file_type": "json",
        "observation_start": START,
        "observation_end": END,
    }
    data = get_json(FRED_URL, params=params)
    if data is None:
        print(f"{series_id} download failed. Check the key in .env and the network connection.")
        continue
    save_json(data, f"fred_{series_id}.json")
    obs = pd.DataFrame(data["observations"])
    obs[column] = pd.to_numeric(obs["value"], errors="coerce")
    fred_tables.append(obs[["date", column]].set_index("date"))
    print(f"{series_id}: {len(obs)} rows, missing values: {obs[column].isna().sum()}")
    time.sleep(1)

FRED key loaded: True | length: 32 (expected: 32)
  Saved data/raw/fred_DEXCHUS.json
DEXCHUS: 1000 rows, missing values: 41
  Saved data/raw/fred_DEXHKUS.json
DEXHKUS: 1000 rows, missing values: 41


In [5]:
fred = pd.concat(fred_tables, axis=1).reset_index()
fred["date"] = pd.to_datetime(fred["date"])
fred["fred_hkd_cny"] = fred["cny_per_usd"] / fred["hkd_per_usd"]

print("Date range:", fred["date"].min().date(), "to", fred["date"].max().date())
print("Duplicate dates:", fred["date"].duplicated().sum())

both = fx.merge(fred[["date", "fred_hkd_cny"]], on="date").dropna()
gap = (both["fred_hkd_cny"] / both["hkd_cny"] - 1).abs() * 100
print("\nDates with both rates:", len(both), "days")
print(f"Gap: mean {gap.mean():.3f}%, maximum {gap.max():.3f}% (expected mean below 0.5%)")

fred.to_csv(RAW / "fx_fred_daily.csv", index=False)
print("\nSaved data/raw/fx_fred_daily.csv")


Date range: 2023-01-02 to 2026-10-30
Duplicate dates: 0

Dates with both rates: 944 days
Gap: mean 0.051%, maximum 0.709% (expected mean below 0.5%)

Saved data/raw/fx_fred_daily.csv


In [6]:
import yfinance as yf

hsi = yf.Ticker("^HSI").history(start=START, end="2026-11-01", auto_adjust=False)

if hsi.empty:
    print("No data returned. Yahoo may be rate-limiting requests. Try again in a few minutes.")
else:
    hsi.index = hsi.index.tz_localize(None).normalize()
    hsi = hsi[["Open", "High", "Low", "Close", "Volume"]].reset_index()
    hsi = hsi.rename(columns={"Date": "date"})

    print("Rows:", len(hsi), "(expected: 920–960 Hong Kong trading days)")
    print("Date range:", hsi["date"].min().date(), "to", hsi["date"].max().date())
    print("Closing price range:", round(hsi["Close"].min()), "to", round(hsi["Close"].max()))
    print(hsi.head())

    hsi.to_csv(RAW / "hsi_daily.csv", index=False)
    print("\nSaved data/raw/hsi_daily.csv")

Rows: 939 (expected: 920–960 Hong Kong trading days)
Date range: 2023-01-03 to 2026-10-30
Closing price range: 14961 to 27968
        date          Open          High           Low         Close  \
0 2023-01-03  19570.429688  20212.550781  19303.730469  20145.289062   
1 2023-01-04  20319.980469  20793.109375  20233.390625  20793.109375   
2 2023-01-05  21295.869141  21396.089844  20962.400391  21052.169922   
3 2023-01-06  21220.890625  21282.710938  20862.769531  20991.640625   
4 2023-01-09  21295.939453  21470.689453  21216.919922  21388.339844   

       Volume  
0  2286700000  
1  3016000000  
2  2875400000  
3  2754500000  
4  2697900000  

Saved data/raw/hsi_daily.csv


/Users/rhea/Desktop/northbound_project/.venv/lib/python3.14/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)


In [7]:
import requests

CSD_URL = "https://www.censtatd.gov.hk/api/get.php"
CSD_PARAMS = {"id": "620-67001", "lang": "en", "full_series": "1"}
CSD_HEADERS = {"User-Agent": "Python Request"}

try:
    response = requests.get(CSD_URL, params=CSD_PARAMS, headers=CSD_HEADERS, timeout=60)
    response.raise_for_status()
    data = response.json()
except (requests.exceptions.RequestException, ValueError) as error:
    print("Download failed:", type(error).__name__)
    data = None

if data is not None:
    save_json(data, "csd_620-67001.json")
    print("Top-level keys:", list(data.keys()))

    retail_raw = pd.DataFrame(data["dataSet"])
    print("\nRows:", len(retail_raw))
    print("Columns:", retail_raw.columns.tolist())
    print("\nFirst 5 rows:")
    print(retail_raw.head().to_string())

    print("\nDistinct text values for choosing filters:")
    for col in retail_raw.columns:
        values = retail_raw[col].astype(str).unique()
        if len(values) <= 15:
            print(f"  {col}: {list(values)}")

  Saved data/raw/csd_620-67001.json
Top-level keys: ['header', 'dataSet']

Rows: 1430
Columns: ['freq', 'period', 'sv', 'svDesc', 'figure', 'sd_value']

First 5 rows:
  freq  period      sv       svDesc  figure sd_value
0    Y    2004  VAL_RS  HK$ million             N.A.
1    M  200410  VAL_RS  HK$ million   15831         
2    M  200411  VAL_RS  HK$ million   15134         
3    M  200412  VAL_RS  HK$ million   18025         
4    Y    2005  VAL_RS  HK$ million  204372         

Distinct text values for choosing filters:
  freq: ['Y', 'M']
  sv: ['VAL_RS', 'VOL_IDX_RS', 'VAL_IDX_RS']
  svDesc: ['HK$ million', 'Index', 'Year-on-year % change']
  sd_value: ['N.A.', '', 'p']


In [8]:
retail = retail_raw[
    (retail_raw["freq"] == "M")
    & (retail_raw["sv"] == "VAL_RS")
    & (retail_raw["svDesc"] == "HK$ million")
].copy()

retail["month"] = pd.to_datetime(retail["period"], format="%Y%m")
retail["retail_value_hkd_mn"] = pd.to_numeric(retail["figure"], errors="coerce")
retail["provisional"] = retail["sd_value"] == "p"
retail = retail[retail["month"] >= START]
retail = retail[["month", "retail_value_hkd_mn", "provisional"]].sort_values("month")

print("Rows:", len(retail), "(expected: 44–46 monthly rows from 2023-01)")
print("Month range:", retail["month"].min().strftime("%Y-%m"), "to", retail["month"].max().strftime("%Y-%m"))
print("Duplicate months:", retail["month"].duplicated().sum())
print("Missing values:", retail["retail_value_hkd_mn"].isna().sum())
print("Retail sales range (HKD millions):", retail["retail_value_hkd_mn"].min(), "to", retail["retail_value_hkd_mn"].max())
print("Months with provisional figures:", retail.loc[retail["provisional"], "month"].dt.strftime("%Y-%m").tolist())
print(retail.tail())

retail.to_csv(RAW / "csd_retail_monthly.csv", index=False)
print("\nSaved data/raw/csd_retail_monthly.csv")


Rows: 45 (expected: 44–46 monthly rows from 2023-01)
Month range: 2023-01 to 2026-09
Duplicate months: 0
Missing values: 0
Retail sales range (HKD millions): 28876 to 37327
Months with provisional figures: ['2026-09']
         month  retail_value_hkd_mn  provisional
281 2026-05-01                33790        False
282 2026-06-01                31490        False
283 2026-07-01                31020        False
284 2026-08-01                32076        False
285 2026-09-01                33357         True

Saved data/raw/csd_retail_monthly.csv
